In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
#from sklearn.cluster import KMeans
from PIL import Image
import os
import pandas as pd
import numpy as np
#from sklearn.cluster import KMeans
import shutil
from collections import Counter
import json
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

In [2]:
# load resNet 50 model
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
def extract_features(path):
    img = Image.open(path).convert('RGB')
    t = transform(img).unsqueeze(0)
    with torch.no_grad():
        return model(t).flatten().numpy()


# paths
samples = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\k-means_training\fabrics'
fabric_csv = "fabric_analysis_results.csv"

print("learning fabrics...")
X_train = []
y_train = []

for fabric_type in os.listdir(samples):
    folder_path = os.path.join(samples, fabric_type)
    
    if os.path.isdir(folder_path):
        print(f"Processing {fabric_type}...")
        for img_name in tqdm(os.listdir(folder_path)):
            if img_name.lower().endswith(('.png', 'jpg', '.jpeg')):
                all_features = extract_features(os.path.join(folder_path, img_name))
                X_train.append(all_features)
                y_train.append(fabric_type)

learning fabrics...
Processing Acrylic...


100%|██████████| 12/12 [00:00<?, ?it/s]


Processing africa_fabric...


100%|██████████| 1059/1059 [01:46<00:00,  9.99it/s]


Processing Artificial_fur...


...

100%|██████████| 11/11 [00:00<?, ?it/s]


Processing Viscose...


100%|██████████| 37/37 [00:00<?, ?it/s]


Processing Wool...


100%|██████████| 90/90 [00:00<00:00, 90006.52it/s]


In [5]:
# SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

clf = SVC(kernel='linear', class_weight='balanced', random_state=42)
clf.fit(X_train_scaled, y_train)
print(f'DONE TRAINING {len(X_train)} images')

#predicitions pleaseeee
print('testing...')
runway = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\runway'
results = []

for root, dirs, files in os.walk(runway):
    for img_name in files:
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_paths = os.path.join(root, img_name)

            try:
                curr_feature = extract_features(img_paths)
                curr_scaled = scaler.transform([curr_feature])

                #pred
                pred = clf.predict(curr_scaled)[0]
                brand_name = os.path.basename(root)

                results.append({
                    "brand": brand_name,
                    "image": img_name,
                    "fabric": pred})

            except Exception as e:
                print(f"Skipping {img_name} due to {e}")

DONE TRAINING 1100 images
testing...


In [6]:
# percentage
#extract fabric names
fabric_name = [item['fabric'] for item in results]

counts = Counter(fabric_name)
total = len(fabric_name)
print("\n--- FINAL FABRIC TRENDS ---")
for fabric, count in counts.items():
    perc = (count/total) * 100
    print(f"{fabric}: {perc:.1f}%")


--- FINAL FABRIC TRENDS ---
mesh: 11.2%
silk: 76.0%
denim: 8.7%
africa_fabric: 4.1%


In [ ]:
import shutil
import os

# 1. Define where you want the sorted folders to go
output_visual_dir = r"C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\outputs\fabric_check"

print("Organizing images into fabric folders for visual verification...")

for item in results:
    fabric_name = item['fabric']
    img_name = item['image']
    brand_name = item['brand']
    
    # We need the original path to the image to copy it
    # This assumes your runway images are in brand subfolders
    original_path = os.path.join(runway, brand_name, img_name)
    
    # Create the destination folder (e.g., .../fabric_check/silk)
    target_folder = os.path.join(output_visual_dir, fabric_name)
    os.makedirs(target_folder, exist_ok=True)
    
    # Copy the file
    try:
        shutil.copy(original_path, os.path.join(target_folder, img_name))
    except Exception as e:
        print(f"Could not copy {img_name}: {e}")

print(f"Done! Go to: {output_visual_dir}")

Organizing images into fabric folders for visual verification...
Done! Go to: C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\outputs\fabric_check


JSON AND CSV EXTRACT

In [ ]:
df = pd.DataFrame(results)
df.to_csv(fabric_csv, index=False)

# 3. Create JSON for Frontend
# This structure allows you to build a "Brand Selector" in your UI
brand_data = {}
for item in results:
    b = item['brand']
    f = item['fabric']
    if b not in brand_data:
        brand_data[b] = []
    brand_data[b].append(f)

frontend_json = {
    "summary": {f: round((c/total)*100, 2) for f, c in counts.items()},
    "brands": {b: dict(Counter(f_list)) for b, f_list in brand_data.items()},
    "raw": results
}

with open('fabric_analysis.json', 'w') as f:
    json.dump(frontend_json, f, indent=4)

print("fabric analysis done and saved to csv and json paths")